# SEO vs SEA — Organic and Paid Acquisition Analysis

Compares organic search (SEO) and paid search (Google Ads) on the same
terms — volume, revenue, and efficiency — using synthetic Search
Console/GA4-style and Google Ads-style data. See `docs/methodology.md`.

In [1]:
import pandas as pd

seo = pd.read_csv("../data/seo_performance.csv", parse_dates=["date"])
ads = pd.read_csv("../data/google_ads.csv", parse_dates=["date"])
print("seo:", seo.shape, "| ads:", ads.shape)
print("keywords tracked:", seo["keyword"].nunique(), "| campaigns:", ads["campaign"].nunique())


seo: (360, 9) | ads: (276, 9)
keywords tracked: 30 | campaigns: 23


## 1. SEO — does ranking position actually drive CTR?

In [2]:
seo["position_bucket"] = pd.cut(seo["position"], [0,3,10,20,100], labels=["1-3","4-10","11-20","21+"])
ctr_by_bucket = seo.groupby("position_bucket", observed=True).apply(
    lambda g: round(100*g["clicks"].sum()/g["impressions"].sum(), 2), include_groups=False
)
print("CTR by position bucket (%):")
print(ctr_by_bucket)


CTR by position bucket (%):
position_bucket
4-10     12.33
11-20     5.20
21+       3.43
dtype: float64


CTR drops off sharply outside the top 10 — the textbook pattern, and the reason position tracking matters more than impressions alone.

## 2. SEO — which landing pages actually make money

In [3]:
by_lp = seo.groupby("landing_page").agg(
    clicks=("clicks","sum"), organic_sessions=("organic_sessions","sum"), organic_revenue=("organic_revenue","sum")
).sort_values("organic_revenue", ascending=False)
by_lp["revenue_per_session"] = (by_lp["organic_revenue"]/by_lp["organic_sessions"]).round(2)
by_lp


## 3. Paid — ROAS by campaign type

In [4]:
ads["campaign_type"] = ads["campaign"].str.rsplit(" - ", n=1).str[0]
by_type = ads.groupby("campaign_type").agg(cost=("cost","sum"), conversion_value=("conversion_value","sum"))
by_type["roas"] = (by_type["conversion_value"]/by_type["cost"]).round(2)
by_type.sort_values("roas", ascending=False)


Brand search converts at close to 31x — expected, it's capturing demand that already exists. Generic search sits at roughly 6.5x, the segment worth testing and iterating on, not the one to cut.

## 4. Organic vs paid, side by side

In [5]:
organic_revenue = seo["organic_revenue"].sum()
organic_sessions = seo["organic_sessions"].sum()
paid_cost = ads["cost"].sum()
paid_revenue = ads["conversion_value"].sum()

print(f"Organic: {organic_sessions:,.0f} sessions, EUR {organic_revenue:,.0f} revenue, EUR 0 spend")
print(f"Paid:    EUR {paid_cost:,.0f} spend, EUR {paid_revenue:,.0f} revenue, {paid_revenue/paid_cost:.1f}x blended ROAS")
print(f"\nOrganic produces {organic_revenue/paid_revenue:.1f}x the revenue of paid, at zero marginal media cost.")


Organic: 28,339 sessions, EUR 562,130 revenue, EUR 0 spend
Paid:    EUR 26,360 spend, EUR 328,220 revenue, 12.5x blended ROAS

Organic produces 1.7x the revenue of paid, at zero marginal media cost.


## 5. Key findings

1. CTR by position bucket: 12.3% (top 10) vs 5.2% (11-20) vs 3.4% (21+) —
   ranking improvements outside the top 10 have a real, measurable payoff
   even before reaching page 1 in the strictest sense.
2. Landing pages differ a lot in revenue per session, not just traffic —
   some high-click pages under-monetise relative to lower-traffic pages,
   which points to on-page conversion issues rather than a traffic problem.
3. Search - Brand ROAS (30.7x) dwarfs Search - Generic (6.5x) and Shopping
   (8.5x). Brand budget is close to riskless; Generic and Shopping are where
   testing and optimisation budget should go, because that's where the
   ROAS is actually movable.
4. Organic revenue is roughly 1.7x paid revenue in this dataset, at zero
   incremental media spend — the case for sustained SEO investment, not a
   reason to under-invest in paid, since paid opens up demand SEO can't
   reach yet (new keywords, competitor terms, remarketing).